# 🧪 Pruebas de Software — Sesiones 6 y 7
## Regression Testing · Smoke Testing · Acceptance Testing

---

**Universidad:** Universitaria de Colombia  
**Materia:** Pruebas de Software  
**Semestre:** 7mo — Ingeniería de Software  
**Docente:** Mg. Sergio Alejandro Burbano Mena  

---

### 📌 ¿Qué aprenderás en este notebook?

| # | Tema | Tiempo estimado |
|---|------|-----------------|
| 1 | Configuración del entorno | 5 min |
| 2 | Sistema de ejemplo: Biblioteca Universitaria | 10 min |
| 3 | **Regression Testing** — Pruebas de Regresión | 25 min |
| 4 | **Smoke Testing** — Pruebas de Humo | 20 min |
| 5 | **Acceptance Testing** — Pruebas de Aceptación (BDD/Gherkin) | 25 min |
| 6 | Integración: los 3 tipos en un pipeline | 10 min |

---

> 💡 **Cómo usar este notebook:**  
> Ejecuta las celdas **en orden, de arriba hacia abajo**.  
> Lee cada comentario antes de ejecutar — el código está diseñado para entenderse leyéndolo.

---
# ⚙️ SECCIÓN 0 — Configuración del Entorno

Instalamos las librerías necesarias para ejecutar los tres tipos de prueba en Colab.

In [11]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 0.1 — Instalación de dependencias
# ═══════════════════════════════════════════════════════════════════

# 'pytest'      : el framework de pruebas más popular de Python
# 'behave'      : framework para BDD (Behavior Driven Development)
#                 permite escribir pruebas de aceptación en Gherkin
# 'pytest-mock' : permite crear objetos falsos (mocks) en los tests
# 'requests'    : para simular llamadas HTTP en el smoke testing
# 'ipytest'     : permite ejecutar pytest directamente en Colab

!pip install pytest behave pytest-mock requests ipytest --quiet

# Mostramos un mensaje de confirmación cuando todo esté instalado
print("✅ Todas las dependencias instaladas y extensión 'ipytest' cargada correctamente.")
print("   pytest     → framework de pruebas")
print("   behave     → BDD / Gherkin para acceptance tests")
print("   pytest-mock → mocks y stubs")
print("   ipytest    → ejecutar pytest en Colab")

✅ Todas las dependencias instaladas y extensión 'ipytest' cargada correctamente.
   pytest     → framework de pruebas
   behave     → BDD / Gherkin para acceptance tests
   pytest-mock → mocks y stubs
   ipytest    → ejecutar pytest en Colab


In [8]:
# Aseguramos que ipytest está instalado antes de configurar
!pip install ipytest --quiet
print("✅ ipytest instalado (por si acaso).")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 58.9 MB/s eta 0:00:00
✅ ipytest instalado (por si acaso).


In [10]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 0.2 — Configuración de ipytest para ejecutar en Colab
# ═══════════════════════════════════════════════════════════════════

# ipytest nos permite usar la magia %%ipytest en las celdas
# para ejecutar tests de pytest directamente en el notebook
import ipytest

# autoconfig() configura ipytest para que funcione automáticamente
# con el nombre del módulo actual del notebook
ipytest.autoconfig()

# Importamos las librerías estándar que usaremos a lo largo del notebook
import pytest          # framework principal de testing
import datetime        # para manejar fechas en los ejemplos
from unittest.mock import MagicMock, patch  # para crear objetos falsos
from dataclasses import dataclass, field    # para crear clases de datos limpias
from typing import List, Optional           # para type hints

print("✅ Configuración lista. Puedes empezar el notebook.")

✅ Configuración lista. Puedes empezar el notebook.


---
# 📊 SECCIÓN de actividad resuelta test


## test de actividad resuelto del archivo word (entregale ) solucionado
:

en estas ultimas tres modulos esta la actividad propuesta de los ejercicios
del word

In [21]:
import pytest
import datetime
from unittest.mock import MagicMock, patch
from dataclasses import dataclass, field
from typing import List, Optional

# Re-import all the models and business logic from the main notebook
# This ensures the tests are running against the latest version of the code

# Models
@dataclass
class Libro:
    id: int
    titulo: str
    autor: str
    disponible: bool = True

@dataclass
class Estudiante:
    codigo: str
    nombre: str
    activo: bool = True
    multa_pendiente: float = 0.0

@dataclass
class Prestamo:
    id: int
    libro_id: int
    estudiante_codigo: str
    fecha_prestamo: datetime.date
    fecha_limite: datetime.date
    devuelto: bool = False

# Constants
DIAS_PRESTAMO = 14
TARIFA_DIARIA_MULTA = 500
LIMITE_PRESTAMOS_ACTIVOS = 3

# Functions
def esta_disponible(libro: Libro) -> bool:
    return libro.disponible

def buscar_libro_por_titulo(catalogo: List[Libro], titulo: str) -> Optional[Libro]:
    for libro in catalogo:
        if titulo.lower() in libro.titulo.lower():
            return libro
    return None

def registrar_prestamo(
    libro: Libro,
    estudiante: Estudiante,
    prestamos_activos: List[Prestamo],
    ultimo_id: int = 1
) -> Prestamo:
    if not libro.disponible:
        raise ValueError(f"El libro '{libro.titulo}' no está disponible")
    if not estudiante.activo:
        raise ValueError(
            f"El estudiante '{estudiante.nombre}' tiene multa pendiente "
            f"de ${estudiante.multa_pendiente:,.0f} COP. "
            "Debe pagarla antes de hacer nuevos préstamos."
        )
    prestamos_sin_devolver = [p for p in prestamos_activos if not p.devuelto]
    if len(prestamos_sin_devolver) >= LIMITE_PRESTAMOS_ACTIVOS:
        raise ValueError(
            f"El estudiante ya tiene {LIMITE_PRESTAMOS_ACTIVOS} préstamos activos. "
            "Debe devolver uno antes de solicitar otro."
        )

    hoy = datetime.date.today()
    limite = hoy + datetime.timedelta(days=DIAS_PRESTAMO)

    nuevo_prestamo = Prestamo(
        id=ultimo_id,
        libro_id=libro.id,
        estudiante_codigo=estudiante.codigo,
        fecha_prestamo=hoy,
        fecha_limite=limite
    )
    libro.disponible = False
    return nuevo_prestamo

def registrar_devolucion(
    prestamo: Prestamo,
    libro: Libro,
    fecha_devolucion: Optional[datetime.date] = None
) -> float:
    if prestamo.devuelto:
        raise ValueError(f"El préstamo #{prestamo.id} ya fue registrado como devuelto")
    if fecha_devolucion is None:
        fecha_devolucion = datetime.date.today()

    diferencia = fecha_devolucion - prestamo.fecha_limite
    dias_retraso = diferencia.days

    multa = max(0, dias_retraso) * TARIFA_DIARIA_MULTA

    prestamo.devuelto = True
    libro.disponible = True
    return multa

def calcular_multa_por_dias(dias_retraso: int) -> float:
    if dias_retraso < 0:
        raise ValueError(f"Los días de retraso no pueden ser negativos: {dias_retraso}")
    return dias_retraso * TARIFA_DIARIA_MULTA

def aplicar_multa_a_estudiante(estudiante: Estudiante, monto: float) -> None:
    estudiante.multa_pendiente += monto
    if estudiante.multa_pendiente > 0:
        estudiante.activo = False

def pagar_multa(estudiante: Estudiante, monto_pago: float) -> float:
    if monto_pago <= 0:
        raise ValueError("El monto de pago debe ser mayor a cero")
    estudiante.multa_pendiente = max(0.0, estudiante.multa_pendiente - monto_pago)
    if estudiante.multa_pendiente == 0.0:
        estudiante.activo = True
    return estudiante.multa_pendiente

def autenticar_usuario(codigo: str, contrasena: str) -> Optional[Estudiante]:
    usuarios_validos = {
        "2024-001": ("pass123", Estudiante("2024-001", "Ana García")),
        "2024-002": ("clave456", Estudiante("2024-002", "Carlos López")),
        "2024-003": ("test789", Estudiante("2024-003", "María Rodríguez")),
    }
    if codigo in usuarios_validos:
        contrasena_correcta, estudiante = usuarios_validos[codigo]
        if contrasena == contrasena_correcta:
            return estudiante
    return None

In [22]:
%%ipytest -v

@pytest.mark.regression
class TestPrestamosRegression:

    def setup_method(self):
        self.libro_disponible = Libro(1, "El Señor de los Anillos", "J.R.R. Tolkien")
        self.estudiante_activo = Estudiante("E001", "Frodo Bolsón")
        self.prestamos_vacios = []

    def test_libro_prestado_no_esta_disponible(self):
        """
        REGRESION: Verifica que un libro actualmente prestado
        no aparece como disponible.
        """
        # ARRANGE
        # libro_disponible y estudiante_activo están en setup_method

        # ACT
        registrar_prestamo(self.libro_disponible, self.estudiante_activo, self.prestamos_vacios)

        # ASSERT
        assert not self.libro_disponible.disponible, \
            "El libro debería estar marcado como no disponible después de ser prestado."


======================================= test session starts ========================================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: mock-3.15.1, anyio-4.13.0, langsmith-0.7.30, typeguard-4.5.1
collected 1 item

t_41a621d70b7a4640afb008f6da44a864.py .                                                      [100%]

======================================== 1 passed in 0.01s =========================================


```gherkin
Feature: Sistema de multas por devolución tardía
  Como estudiante de la Universitaria de Colombia
  Quiero conocer y gestionar mis multas de biblioteca
  Para poder planificar mis reservas de libros

  # AC-1: Cálculo correcto de la multa
  Scenario: Cálculo de multa por 3 días de retraso
    Given un estudiante tiene un libro prestado
    And la fecha límite de devolución fue hace 3 días
    When el estudiante devuelve el libro hoy
    Then el sistema calcula una multa de $1500 COP

  # AC-2: Mensaje cuando no hay multa
  Scenario: Estudiante sin multa pendiente
    Given un estudiante no tiene multas pendientes
    When el estudiante consulta su estado de multas
    Then el sistema le informa "Sin multa pendiente"

  # AC-3: Bloqueo por multa pendiente
  Scenario: Estudiante con multa no puede reservar
    Given un estudiante tiene una multa pendiente
    And el estudiante está bloqueado para nuevos préstamos
    When el estudiante intenta reservar un libro
    Then el sistema rechaza el préstamo
    And el sistema le informa que tiene una multa pendiente
```

In [23]:
import json

class MockResponse:
    """A mock response object to simulate requests.Response"""
    def __init__(self, json_data, status_code):
        self._json_data = json_data
        self.status_code = status_code

    def json(self):
        return self._json_data

    @property
    def text(self):
        return json.dumps(self._json_data)

class MockRequests:
    """A mock requests library to simulate API calls based on a predefined state."""
    def __init__(self, simulator_estado='ok'):
        self.simulator_estado = simulator_estado
        self.responses = {
            '/health': MockResponse({'status': 'ok'}, 200),
            '/health/db': MockResponse({'db_status': 'connected'}, 200),
            '/auth/login': MockResponse({'access_token': 'fake_token'}, 200),
            '/libros': MockResponse([], 200), # Empty list of books
            '/multas/2024-001': MockResponse({'multa_pendiente': 0.0}, 200)
        }

    def get(self, url, **kwargs):
        path = url.replace(BASE, '') # Extract the path from the full URL
        return self._get_response(path)

    def post(self, url, json=None, data=None, **kwargs):
        path = url.replace(BASE, '') # Extract the path from the full URL
        if path == '/auth/login':
            # Simulate authentication logic
            if json and json.get('email') == 'test@uni.edu.co' and json.get('password') == 'Test123!':
                return self.responses[path]
            else:
                return MockResponse({'detail': 'Invalid credentials'}, 401)
        return self._get_response(path)

    def _get_response(self, path):
        if self.simulator_estado == 'error':
            return MockResponse({'detail': 'Internal Server Error'}, 500)
        if path in self.responses:
            return self.responses[path]
        return MockResponse({'detail': 'Not Found'}, 404)

# Define BASE URL for mock requests
BASE = 'http://localhost:8000/api' # This URL is just a placeholder for the mock

In [24]:
%%ipytest -v

import pytest
import requests
from unittest.mock import patch # Needed here for the patch decorators

@pytest.mark.smoke
class TestBibliotecaSmoke:

    api_mock = None # Placeholder for our mocked requests instance

    @classmethod
    def setup_class(cls):
        # Initialize the mock requests for the entire test class
        cls.api_mock = MockRequests(simulator_estado='ok')

    @patch('requests.get', side_effect=lambda url, **kwargs: TestBibliotecaSmoke.api_mock.get(url, **kwargs))
    @patch('requests.post', side_effect=lambda url, json=None, data=None, **kwargs: TestBibliotecaSmoke.api_mock.post(url, json=json, data=data, **kwargs))
    def test_01_api_disponible(self, mock_get, mock_post):
        """
        SMOKE: Verificación de disponibilidad del servidor.
        Llama al endpoint /api/health y espera un status 200 y 'status': 'ok'.
        """
        response = requests.get(f'{BASE}/health')
        assert response.status_code == 200
        assert response.json()['status'] == 'ok'

    @patch('requests.get', side_effect=lambda url, **kwargs: TestBibliotecaSmoke.api_mock.get(url, **kwargs))
    @patch('requests.post', side_effect=lambda url, json=None, data=None, **kwargs: TestBibliotecaSmoke.api_mock.post(url, json=json, data=data, **kwargs))
    def test_02_login_funciona(self, mock_get, mock_post):
        """
        SMOKE: Verificación de la funcionalidad de login.
        Llama al endpoint /api/auth/login con credenciales válidas y espera un status 200 y un 'access_token'.
        """
        response = requests.post(f'{BASE}/auth/login', json={'email': 'test@uni.edu.co', 'password': 'Test123!'})
        assert response.status_code == 200
        assert 'access_token' in response.json()

    @patch('requests.get', side_effect=lambda url, **kwargs: TestBibliotecaSmoke.api_mock.get(url, **kwargs))
    @patch('requests.post', side_effect=lambda url, json=None, data=None, **kwargs: TestBibliotecaSmoke.api_mock.post(url, json=json, data=data, **kwargs))
    def test_03_listar_libros_responde(self, mock_get, mock_post):
        """
        SMOKE: Verificación del catálogo de libros.
        Llama al endpoint /api/libros y espera un status 200 (OK) o 401 (Unauthorized).
        """
        response = requests.get(f'{BASE}/libros')
        assert response.status_code in [200, 401]

    @patch('requests.get', side_effect=lambda url, **kwargs: TestBibliotecaSmoke.api_mock.get(url, **kwargs))
    @patch('requests.post', side_effect=lambda url, json=None, data=None, **kwargs: TestBibliotecaSmoke.api_mock.post(url, json=json, data=data, **kwargs))
    def test_04_base_de_datos_conectada(self, mock_get, mock_post):
        """
        SMOKE: Verificación de la conexión a la base de datos.
        Llama al endpoint /api/health/db y espera un status 200 y 'db_status': 'connected'.
        """
        response = requests.get(f'{BASE}/health/db')
        assert response.status_code == 200
        assert response.json()['db_status'] == 'connected'

    @patch('requests.get', side_effect=lambda url, **kwargs: TestBibliotecaSmoke.api_mock.get(url, **kwargs))
    @patch('requests.post', side_effect=lambda url, json=None, data=None, **kwargs: TestBibliotecaSmoke.api_mock.post(url, json=json, data=data, **kwargs))
    def test_05_modulo_multas_responde(self, mock_get, mock_post):
        """
        SMOKE: Verificación de la disponibilidad del módulo de multas.
        Llama al endpoint /api/multas/2024-001 y espera que no retorne 404 ni 500.
        """
        response = requests.get(f'{BASE}/multas/2024-001')
        assert response.status_code not in [404, 500]


======================================= test session starts ========================================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: mock-3.15.1, anyio-4.13.0, langsmith-0.7.30, typeguard-4.5.1
collected 5 items

t_41a621d70b7a4640afb008f6da44a864.py .....                                                  [100%]

======================================== 5 passed in 0.03s =========================================
